# IPO-Ready · Two pre-IPO diligence cases

Cutoff **25 November 2026**. This notebook reproduces the two live cases from the Python-built company model and retained evidence. Run `python -X utf8 -m ipo_ready.pipeline --offline` from the repository root first. Open the linked primary source pages to verify statement scope, and the free-research audit to repeat the information-gap comparison.

The company data values are normalized to · crore unless noted for share prices, earnings per share, transaction fees and trade volume. Operating cash sensitivities are historical calculations. The dealer indications and share assumptions are dated illustrations. [Evidence and free-research audit](../docs/research/topics/v21-pre-ipo-cases.md)

In [1]:
from decimal import Decimal as D
from pathlib import Path
import json

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "ipo_ready" / "pre_ipo.py").exists())
data = root / "site" / "data" / "current"
manifest = json.loads((data / "manifest.json").read_text(encoding="utf-8"))
company_ids = ("igx", "oyo", "nse-iex", "nse-mcx", "nse-lemontree")
companies = {name: json.loads((data / "company" / f"{name}.json").read_text(encoding="utf-8"))
             for name in company_ids}
print(f"Offline build {manifest['build_id']} · Python pipeline {manifest['pipeline_version']}")
for name in ("igx", "oyo"):
    issuer = companies[name]
    status = issuer["ipo_stage"]
    evidence = issuer["evidence"][status["ev"]]
    print(f"{issuer['short']}: {status['label']} · {status['milestone']} · {evidence['url']}")


Offline build 20261125T065438Z · Python pipeline 0.10.0
IGX: Pre-IPO · SEBI observations received · 12 Oct 2026 · https://nsearchives.nseindia.com/corporate/IEX_13102026132246_IEX_Intimation_IGX_DRHP_SEBI_Observation_Letter_Signed.pdf#page=1
OYO: Pre-IPO · Updated draft · October addendum · https://www.sebi.gov.in/sebi_data/attachdocs/oct-2026/1792987300020.pdf#page=1


## 1 · What drives growth?

IGX sells gas-market access and transaction services. Apply FY2025's observed average fee per unit to the change in volumes, and attribute the residual to the rate and mix together. Oravel is a global hospitality platform. Use the contribution of G6 to the change in group gross booking value, with revenue growth shown independently.

In [2]:
igx, oyo = companies["igx"], companies["oyo"]
fy = "FY2026"
prior = "FY2025"

def value(cell):
    return D(cell["value"])

def facts(company, period):
    return company["analysis"][period]["facts"]

def evidence_for(company, index):
    return company["evidence"][index]

activity = igx["market_activity"]
vol_26, vol_25 = (value(activity[p]["volume"]) for p in (fy, prior))
fees_26, fees_25 = (value(activity[p]["fees"]) for p in (fy, prior))
volume_effect = (vol_26 - vol_25) * fees_25 / vol_25
rate_mix = fees_26 - fees_25 - volume_effect
assert volume_effect + rate_mix == fees_26 - fees_25
print(f"IGX: {vol_25} → {vol_26} million MMBtu. Fee increase ₹{fees_26-fees_25}m.")
print(f"Volume effect ₹{volume_effect:.3f}m · rate/mix effect ₹{rate_mix:.3f}m · FY26 rate ₹{fees_26/vol_26:.4f}/MMBtu.")

# Use the native unit from the published evidence for booking values.
def raw(company, page, label, period):
    item = next(item for item in company["evidence"].values() if item.get("page") == page
                and item.get("period") == period and item.get("raw_label") == label)
    return D(item["raw_text"].replace(",", "").replace("(", "-").replace(")", ""))

gbv = raw(oyo, 9, "Gross Booking Value", fy) - raw(oyo, 9, "Gross Booking Value", prior)
g6 = raw(oyo, 9, "G6 GBV", fy) - raw(oyo, 9, "G6 GBV", prior)
revenue_growth = (value(facts(oyo, fy)["revenue_ops"]) / value(facts(oyo, prior)["revenue_ops"]) - 1) * 100
print(f"Oravel: operating revenue +{revenue_growth:.2f}%. G6 contributed {100*g6/gbv:.2f}% of incremental bookings.")
assert D(oyo["issuer_case"]["metrics"]["g6_pct_of_gbv_increase"]) == g6 / gbv * 100


IGX: 60.02 → 76.79 million MMBtu. Fee increase ₹125.67m.
Volume effect ₹125.747m · rate/mix effect ₹-0.077m · FY26 rate ₹7.4973/MMBtu.
Oravel: operating revenue +49.66%. G6 contributed 73.44% of incremental bookings.


## 2 · Is profit turning into cash?

For IGX remove one grouped working-capital line to see how large that line is relative to CFO. It contains several liabilities and the remaining amount is a sensitivity. For Oravel subtract paid capex, borrowing interest and lease principal and interest. The result excludes debt principal and acquisitions. Compare tax credits with the reported FY2026 profit, and check the foreign-exchange effect separately.

In [3]:
def cash(company, period):
    return company["cash_flow"]["by_period"][period]["lines"]

for period in ("FY2024", prior, fy):
    lines = cash(igx, period)
    assert sum(value(lines[k]) for k in ("cfo", "cfi", "cff")) == value(lines["net_change"])
    assert value(lines["opening"]) + value(lines["net_change"]) == value(lines["closing"])
igx_cash = cash(igx, fy)
cash_sensitivity = value(igx_cash["cfo"]) - value(igx_cash["payables_and_liabilities"])
assert cash_sensitivity == D("18.057")
print(f"IGX FY2026 CFO less grouped liabilities: ₹{cash_sensitivity} crore.")

oyo_cash = cash(oyo, fy)
cash_keys = ("cfo", "capex", "debt_interest", "lease_principal", "lease_interest")
for period in (prior, fy):
    lines = cash(oyo, period)
    cash_after = sum(value(lines[key]) for key in cash_keys)
    print(f"Oravel {period} cash after capex, borrowing interest and lease payments: ₹{cash_after} crore.")
assert sum(value(oyo_cash[key]) for key in cash_keys) == D("408.465")
assert sum(value(oyo_cash[k]) for k in ("opening", "net_change", "other_effects")) == value(oyo_cash["closing"])
assert oyo["cash_flow"]["by_period"][prior]["reconciliation"]["state"] == "rounding_difference"
print(f"FY2026 net income-tax credit: ₹{-value(facts(oyo, fy)['tax'])} crore.")


IGX FY2026 CFO less grouped liabilities: ₹18.057 crore.
Oravel FY2025 cash after capex, borrowing interest and lease payments: ₹-1038.572 crore.
Oravel FY2026 cash after capex, borrowing interest and lease payments: ₹408.465 crore.
FY2026 net income-tax credit: ₹595.251 crore.


## 3 · Which listed benchmarks fit?

Read the explicit scope for each benchmark. This retains IEX for compatible operating measures, MCX for context and Lemon Tree for the third-party hotel fee activity. A matching period and currency alone are insufficient for whole-company operating, cash or valuation comparisons.

In [4]:
for issuer in (igx, oyo):
    for benchmark in issuer["issuer_case"]["benchmarks"]:
        company = companies[benchmark["id"]]
        print(f"{issuer['short']} → {company['short']} ({benchmark['scope']}): {benchmark['rationale']}")
        for name in (issuer, company):
            revenue = name["research"]["by_period"][fy]["comparison"]["revenue"]
            print(f"  {name['short']} {fy} operating revenue: ₹{revenue['value']} crore on {revenue['basis']} basis")


IGX → IEX (operating): Both run energy marketplaces. Use matched annual operating rows. IEX is IGX's promoter and accounts for IGX as an associate in consolidated results.
  IGX FY2026 operating revenue: ₹61.005 crore on issuer_and_esos_trust basis
  IEX FY2026 operating revenue: ₹608 crore on standalone basis
IGX → MCX (context): An exchange operating model offers context. Derivatives, clearing and trade units differ from physically delivered gas.
  IGX FY2026 operating revenue: ₹61.005 crore on issuer_and_esos_trust basis
  MCX FY2026 operating revenue: ₹2154 crore on standalone basis
OYO → LEMONTREE (context): The FY2026 third-party management and franchise fees illustrate one shared business activity. Lemon Tree also owns and leases hotels. The group margins, cash conversion and quote multiples are not peer benchmarks.
  OYO FY2026 operating revenue: ₹9357.984 crore on consolidated basis
  LEMONTREE FY2026 operating revenue: ₹1444.5033 crore on consolidated basis


## 4 · What can we say about valuation?

Use the dated provider indication and reported basic annual EPS. Price times the filed share count is an illustrative equity value. Oravel's converted denominator is a draft scenario before new issuance. Refresh offer prices and capitalization before making a final comparison.

In [5]:
for issuer in (igx, oyo):
    price = D(issuer["quote"]["price_inr"])
    valuation = issuer["valuation"]
    eps = D(valuation["eps_reported_basic"])
    shares = D(valuation["shares_used"])
    equity = price * shares / D("1e7")
    print(f"{issuer['short']}: ₹{price}/share, captured {issuer['quote']['as_of'][:10]}. "
          f"{price/eps:.2f}× basic EPS. ₹{equity:,.2f} crore on {shares:,} draft shares.")
    assert equity.quantize(D("0.0001")) == value(valuation["market_cap"])
assert oyo["valuation"]["market_cap"]["state"] == "conditional_share_count"


IGX: ₹488.00/share, captured 2026-11-23. 85.92× basic EPS. ₹3,660.00 crore on 75,000,000 draft shares.
OYO: ₹20.45/share, captured 2026-11-23. 30.07× basic EPS. ₹30,785.00 crore on 15,053,791,264 draft shares.


## Free research and what remains open

Free Angel One coverage surfaced IGX volume, scale and operating risks. IPO-Ready reproduces the volume and fee bridge, identifies the cash-flow outlier and tests comparable scope in the same workflow. Free StopDown and Priveq coverage surfaced Oravel's proposed issue change, tax sensitivity, capital structure and indicative valuation. Priveq already published the FY2026 cash-after-obligations result. The workbench adds the matched FY2025 calculation, the cash reconciliation and direct source inspection in comparison. For both issuers the cutoff search found draft-stage documents and no confirmed executable off-exchange quote and transfer terms or official offer price. The audit records the reviewed pages and bounded limits. [Free-research audit](../docs/research/topics/v21-pre-ipo-cases.md)